# Prompt Evaluation Workflow

1. **Draft a Prompt**
   - Write the initial prompt and define the expected output.

2. **Create an Evaluation Dataset**
   - Prepare test cases with sample inputs and expected outputs.

3. **Feed Through Claude**
   - Run the prompt through Claude using the evaluation dataset.
   - Collect and record the generated outputs.

4. **Feed Through Grader**
   - Evaluate Claude's outputs against predefined criteria.
   - Measure accuracy, relevance, consistency, and instruction adherence.

5. **Change Prompt and Repeat**
   - Analyze evaluation results and identify weaknesses.
   - Refine the prompt based on the grader's feedback.
   - Repeat the evaluation workflow until the desired performance is achieved.

In [14]:
# load env variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os

In [15]:
# create api client 
from anthropic import Anthropic
client = Anthropic(
    base_url = os.getenv("ANTHROPIC_BASE_URL"),
    api_key = os.getenv("ANTHROPIC_AUTH_TOKEN"),
)

model_name = os.getenv("ANTHROPIC_MODEL")

In [16]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model_name,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [17]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [18]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [19]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [20]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [21]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [22]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [25]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 4.833333333333333


In [27]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Certainly! Below is a JSON object representing a list of 3 Amazon EC2 instances:\n\n```json\n{\n  \"instances\": [\n    {\n      \"instanceId\": \"i-1234567890abcdef0\",\n      \"instanceType\": \"t2.micro\",\n      \"instanceState\": \"running\",\n      \"publicIp\": \"192.0.2.1\",\n      \"privateIp\": \"10.0.0.1\",\n      \"keyName\": \"default\",\n      \"imageId\": \"ami-12345678\",\n      \"securityGroups\": [\"default\"],\n      \"createdTime\": \"2023-10-01T00:00:00Z\",\n      \"userData\": \"{\\\"test\\\":\\\"data\\\"}\",\n      \"iamInstanceProfile\": {\n        \"arn\": \"arn:aws:iam::123456789012:instance-profile/instance-profile-name\"\n      }\n    },\n    {\n      \"instanceId\": \"i-234567890abcdef0\",\n      \"instanceType\": \"t2.micro\",\n      \"instanceState\": \"running\",\n      \"publicIp\": \"192.0.2.2\",\n      \"privateIp\": \"10.0.0.2\",\n      \"keyName\": \"default\",\n      \"imageId\": \"ami-12345678\",\n      \"securityGroups\": [\"